### Model training

In [20]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
import time

warnings.filterwarnings('ignore')

# Optional: nicer display for pandas in notebooks
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)

# Update this path if your CSV is in a different location within the project
csv_path = Path('data/clean/US_Accidents_March23_NY_Clean.csv')
if not csv_path.exists():
    raise FileNotFoundError(f'CSV not found at {csv_path}. Please place the file there or change the path in this cell.')

# Column groups
target_col = 'Severity_Class'
infra_cols = [
    'Amenity','Bump','Crossing','Give_Way','Junction','No_Exit','Railway',
    'Roundabout','Station','Stop','Traffic_Calming','Traffic_Signal','Turning_Loop'
]

numeric_cols = ['Distance(mi)','Temperature(F)','Visibility(mi)',
                        'Humidity(%)','Pressure(in)','Wind_Speed(mph)','Hour','Duration_min']

category_cols = ['Street_encoded', 'County','Weather_Condition','Wind_Direction','Weather_Short','Season']

feature_cols = numeric_cols + category_cols + infra_cols
# Combined categorical columns (infrastructure flags are categorical / indicator features)
use_cols = feature_cols + [target_col]
dtype_map = {
    **{col: 'Int8' for col in infra_cols},
    'Distance(mi)': 'float32',
    'Temperature(F)': 'float32',
    'Visibility(mi)': 'float32',
    'Humidity(%)': 'float32',
    'Pressure(in)': 'float32',
    'Wind_Speed(mph)': 'float32',
    'Hour': 'int16',
    'Duration_min': 'float32',
}

# Read CSV with memory-friendly dtypes
df = pd.read_csv(csv_path, usecols=use_cols, dtype=dtype_map, low_memory=False)
df = df.head(50000)  # For quicker testing during development; remove or adjust as needed
# Cast categoricals to category for compact storage and safer encoding
df[category_cols] = df[category_cols].apply(lambda s: s.astype('category'))
df[target_col] = df[target_col].astype('category')

print('Loaded:', csv_path, '-> shape:', df.shape)
print('Approx memory MB:', round(df.memory_usage(deep=True).sum() / 1e6, 2))

df.head()


Loaded: data\clean\US_Accidents_March23_NY_Clean.csv -> shape: (50000, 28)
Approx memory MB: 3.23


,Distance(mi),County,Temperature(F),Visibility(mi),Pressure(in),Humidity(%),Weather_Condition,Wind_Direction,Wind_Speed(mph),Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Hour,Duration_min,Weather_Short,Season,Severity_Class,Street_encoded
0,0.0100,Rockland,53.099998,1.0,29.809999,93.0,Overcast,East,11.500000,0,0,0,0,0,0,0,0,0,0,0,0,0,16,44.816666,Cloudy/Clear,Autumn,Major,118
1,1.4925,Rockland,53.099998,1.0,29.830000,93.0,Overcast,ENE,8.100000,0,0,0,0,0,0,0,0,0,0,0,0,0,15,89.566666,Cloudy/Clear,Autumn,Major,141
2,1.1700,Westchester,52.000000,2.0,29.809999,93.0,Light Rain,East,17.299999,0,0,0,0,1,0,0,0,0,0,0,0,0,15,89.733330,Rain/Precip,Autumn,Major,1682
3,0.0100,Putnam,52.000000,3.0,29.860001,89.0,Rain,East,10.400000,0,0,0,0,0,0,0,0,0,0,0,0,0,16,44.816666,Rain/Precip,Autumn,Major,1895
4,0.0100,Putnam,52.000000,3.0,29.860001,89.0,Rain,East,10.400000,0,0,0,0,0,0,0,0,0,0,0,0,0,16,44.783333,Rain/Precip,Autumn,Major,1895


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 28 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   Distance(mi)       50000 non-null  float32 
 1   County             50000 non-null  category
 2   Temperature(F)     50000 non-null  float32 
 3   Visibility(mi)     50000 non-null  float32 
 4   Pressure(in)       50000 non-null  float32 
 5   Humidity(%)        50000 non-null  float32 
 6   Weather_Condition  50000 non-null  category
 7   Wind_Direction     50000 non-null  category
 8   Wind_Speed(mph)    50000 non-null  float32 
 9   Amenity            50000 non-null  Int8    
 10  Bump               50000 non-null  Int8    
 11  Crossing           50000 non-null  Int8    
 12  Give_Way           50000 non-null  Int8    
 13  Junction           50000 non-null  Int8    
 14  No_Exit            50000 non-null  Int8    
 15  Railway            50000 non-null  Int8    
 16  Roun

In [22]:
# Combined categorical columns (infrastructure flags are categorical / indicator features)
print('Categorical columns', category_cols + infra_cols)
print('Numeric columns', numeric_cols)
print('Target column', target_col)


Categorical columns ['Street_encoded', 'County', 'Weather_Condition', 'Wind_Direction', 'Weather_Short', 'Season', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop']
Numeric columns ['Distance(mi)', 'Temperature(F)', 'Visibility(mi)', 'Humidity(%)', 'Pressure(in)', 'Wind_Speed(mph)', 'Hour', 'Duration_min']
Target column Severity_Class


In [23]:
df.isna().sum()

Distance(mi)         0
County               0
Temperature(F)       0
Visibility(mi)       0
Pressure(in)         0
Humidity(%)          0
Weather_Condition    0
Wind_Direction       0
Wind_Speed(mph)      0
Amenity              0
Bump                 0
Crossing             0
Give_Way             0
Junction             0
No_Exit              0
Railway              0
Roundabout           0
Station              0
Stop                 0
Traffic_Calming      0
Traffic_Signal       0
Turning_Loop         0
Hour                 0
Duration_min         0
Weather_Short        0
Season               0
Severity_Class       0
Street_encoded       0
dtype: int64

In [24]:
df.head(5)

,Distance(mi),County,Temperature(F),Visibility(mi),Pressure(in),Humidity(%),Weather_Condition,Wind_Direction,Wind_Speed(mph),Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Hour,Duration_min,Weather_Short,Season,Severity_Class,Street_encoded
0,0.0100,Rockland,53.099998,1.0,29.809999,93.0,Overcast,East,11.500000,0,0,0,0,0,0,0,0,0,0,0,0,0,16,44.816666,Cloudy/Clear,Autumn,Major,118
1,1.4925,Rockland,53.099998,1.0,29.830000,93.0,Overcast,ENE,8.100000,0,0,0,0,0,0,0,0,0,0,0,0,0,15,89.566666,Cloudy/Clear,Autumn,Major,141
2,1.1700,Westchester,52.000000,2.0,29.809999,93.0,Light Rain,East,17.299999,0,0,0,0,1,0,0,0,0,0,0,0,0,15,89.733330,Rain/Precip,Autumn,Major,1682
3,0.0100,Putnam,52.000000,3.0,29.860001,89.0,Rain,East,10.400000,0,0,0,0,0,0,0,0,0,0,0,0,0,16,44.816666,Rain/Precip,Autumn,Major,1895
4,0.0100,Putnam,52.000000,3.0,29.860001,89.0,Rain,East,10.400000,0,0,0,0,0,0,0,0,0,0,0,0,0,16,44.783333,Rain/Precip,Autumn,Major,1895


In [25]:
# Work on a copy to be safe (kept as a note, but avoided here to save memory)
# Ensure expected dtypes remain compact
for col in infra_cols:
    if str(df[col].dtype) != 'Int8':
        df[col] = df[col].astype('Int8')

for col in category_cols:
    if df[col].dtype.name != 'category':
        df[col] = df[col].astype('category')

df[target_col] = df[target_col].astype('category')
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 28 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   Distance(mi)       50000 non-null  float32 
 1   County             50000 non-null  category
 2   Temperature(F)     50000 non-null  float32 
 3   Visibility(mi)     50000 non-null  float32 
 4   Pressure(in)       50000 non-null  float32 
 5   Humidity(%)        50000 non-null  float32 
 6   Weather_Condition  50000 non-null  category
 7   Wind_Direction     50000 non-null  category
 8   Wind_Speed(mph)    50000 non-null  float32 
 9   Amenity            50000 non-null  Int8    
 10  Bump               50000 non-null  Int8    
 11  Crossing           50000 non-null  Int8    
 12  Give_Way           50000 non-null  Int8    
 13  Junction           50000 non-null  Int8    
 14  No_Exit            50000 non-null  Int8    
 15  Railway            50000 non-null  Int8    
 16  Roun

In [27]:
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report

In [28]:
# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# --------------------------------------------------------------------------
# 1) Train/test split
# - Stratify by target so class proportions are preserved in train/test
# - This prevents accidental optimistic evaluation when classes are imbalanced
# --------------------------------------------------------------------------
X = df[feature_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)


In [29]:
# --------------------------------------------------------------------------
# 2) Preprocessing pipelines
# - Numeric: median imputation + standard scaling (robust and works for most models)
# - Categorical & infra flags: most-frequent imputation + one-hot encoding
# - Keep preprocessing inside a pipeline to avoid train-test leakage and to make CV safe
# - Sparse-friendly: scaling without centering and sparse OHE to reduce memory
# --------------------------------------------------------------------------

num_cols = numeric_cols + infra_cols

num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler(with_mean=False))  # keep sparse representation
])

cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, category_cols)
], remainder='drop', sparse_threshold=0.0)  # always return sparse matrix


In [30]:
df[category_cols].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   Street_encoded     50000 non-null  category
 1   County             50000 non-null  category
 2   Weather_Condition  50000 non-null  category
 3   Wind_Direction     50000 non-null  category
 4   Weather_Short      50000 non-null  category
 5   Season             50000 non-null  category
dtypes: category(6)
memory usage: 359.0 KB


In [31]:
# --------------------------------------------------------------------------
# 3) Baseline models
# - LogisticRegression: interpretable baseline with class_weight="balanced"
# - DecisionTree: simple nonlinear baseline
# - RandomForest: ensemble baseline capturing interactions and non-linearities
# Each model is wrapped in a pipeline that includes the preprocessor
# - Sparse-friendly configuration for lower memory
# --------------------------------------------------------------------------

lr_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', LogisticRegression(solver='saga', class_weight='balanced', multi_class='multinomial', max_iter=500, n_jobs=4, random_state=RANDOM_SEED))
])

dt_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_SEED))
])

rf_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=4))
])


In [32]:
# Fit baselines and print evaluation (weighted F1 is primary metric)
print("Fitting baseline models...")
for name, pipe in [('LogisticRegression', lr_pipeline), ('DecisionTree', dt_pipeline), ('RandomForest', rf_pipeline)]:
    pipe.fit(X_train, y_train)                       # train model
    preds = pipe.predict(X_test)                     # predict on test set
    print(f"\n=== {name} ===")
    print("Weighted F1:", f1_score(y_test, preds, average='weighted'))  # primary metric
    print(classification_report(y_test, preds))     # precision/recall per class;

Fitting baseline models...

=== LogisticRegression ===
Weighted F1: 0.8167527215339317
              precision    recall  f1-score   support

       Major       0.75      0.92      0.83      4421
       Minor       0.00      0.00      0.00       104
    Moderate       0.91      0.76      0.82      5475

    accuracy                           0.82     10000
   macro avg       0.55      0.56      0.55     10000
weighted avg       0.83      0.82      0.82     10000


=== DecisionTree ===
Weighted F1: 0.796808365253378
              precision    recall  f1-score   support

       Major       0.78      0.79      0.79      4421
       Minor       0.28      0.32      0.30       104
    Moderate       0.82      0.81      0.81      5475

    accuracy                           0.80     10000
   macro avg       0.63      0.64      0.63     10000
weighted avg       0.80      0.80      0.80     10000


=== RandomForest ===
Weighted F1: 0.8492534900830817
              precision    recall  f1-score 

In [33]:
import xgboost as xgb

In [34]:
# -----------------------
# XGBoost pipeline (preprocessing -> XGB classifier)
# - We use sklearn wrapper XGBClassifier so it fits into Pipeline
# - Use histogram tree builder and modest params to stay memory efficient
# -----------------------
xgb_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', xgb.XGBClassifier(
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=RANDOM_SEED,
        use_label_encoder=False,
        tree_method='hist',
        max_depth=6,
        n_estimators=400,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=4
    ))
])


In [35]:
# -----------------------
# Option: train with SMOTE (example)
# - Use imblearn Pipeline so SMOTE is applied only to training folds
# -----------------------
use_smote = False  # set True to try oversampling
if use_smote:
    xgb_smote_pipeline = ImbPipeline([
        ('preproc', preprocessor),
        ('smote', SMOTE(random_state=RANDOM_SEED)),
        ('clf', xgb.XGBClassifier(objective='multi:softprob', random_state=RANDOM_SEED, use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1))
    ])

In [36]:
# -----------------------
# Quick baseline fit for XGBoost (default params)
# -----------------------
# print("\nFitting XGBoost baseline (default params)...")

# xgb_pipeline.fit(X_train, y_train)
# xgb_preds = xgb_pipeline.predict(X_test)
# print("XGBoost baseline weighted F1:", f1_score(y_test, xgb_preds, average='weighted'))
# print(classification_report(y_test, xgb_preds))

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report

print("\nFitting XGBoost baseline (default params)...")

# encode labels (fit only on training labels)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# fit and predict with encoded labels
xgb_pipeline.fit(X_train, y_train_enc)
xgb_preds_enc = xgb_pipeline.predict(X_test)

# decode predictions back to original string labels for readable reports
xgb_preds = le.inverse_transform(xgb_preds_enc)
y_test_labels = le.inverse_transform(y_test_enc)

print("XGBoost baseline weighted F1:", f1_score(y_test_labels, xgb_preds, average='weighted'))
print(classification_report(y_test_labels, xgb_preds))


Fitting XGBoost baseline (default params)...
XGBoost baseline weighted F1: 0.8754531335438868
              precision    recall  f1-score   support

       Major       0.88      0.86      0.87      4421
       Minor       0.76      0.25      0.38       104
    Moderate       0.87      0.91      0.89      5475

    accuracy                           0.88     10000
   macro avg       0.84      0.67      0.71     10000
weighted avg       0.88      0.88      0.88     10000



In [37]:
# -----------------------
# Hyperparameter tuning (RandomizedSearchCV)
# - StratifiedKFold preserves class proportions across folds
# - Optimize for weighted F1 (primary metric)
# - Keep search compact for speed; expand ranges for full tuning
# - Encodes string labels once to keep XGBoost happy
# -----------------------

import time
from sklearn.preprocessing import LabelEncoder

param_dist = {
    'clf__max_depth': [4, 6, 8],
    'clf__learning_rate': [0.01, 0.03, 0.1],
    'clf__n_estimators': [100, 300, 600],
    'clf__subsample': [0.6, 0.8, 1.0],
    'clf__colsample_bytree': [0.6, 0.8, 1.0]
}

# Encode labels to integers once
le = LabelEncoder()
le.fit(y_train)
y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

# Sample training data for tuning to save memory
sample_n = 80000
if len(X_train) > sample_n:
    X_tune, _, y_tune, _ = train_test_split(
        X_train, y_train, train_size=sample_n, stratify=y_train, random_state=RANDOM_SEED
    )
    y_tune_enc = le.transform(y_tune)
else:
    X_tune, y_tune = X_train, y_train
    y_tune_enc = y_train_enc

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
rand_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist,
    n_iter=5,
    scoring='f1_weighted',
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=1,
    error_score='raise'
)

print("Running RandomizedSearchCV for XGBoost...")
t0 = time.time()
rand_search.fit(X_tune, y_tune_enc)
t1 = time.time()
print(f"RandomizedSearchCV done in {t1 - t0:.1f}s")

best_xgb = rand_search.best_estimator_
y_pred_enc = best_xgb.predict(X_test)
y_pred = le.inverse_transform(y_pred_enc)

print('Best params:', rand_search.best_params_)
print('Best CV weighted F1:', rand_search.best_score_)
from sklearn.metrics import f1_score
print('Test weighted F1:', f1_score(y_test_enc, y_pred_enc, average='weighted'))


Running RandomizedSearchCV for XGBoost...
Fitting 5 folds for each of 5 candidates, totalling 25 fits
RandomizedSearchCV done in 345.0s
Best params: {'clf__subsample': 0.8, 'clf__n_estimators': 600, 'clf__max_depth': 6, 'clf__learning_rate': 0.1, 'clf__colsample_bytree': 1.0}
Best CV weighted F1: 0.8767720588417554
Test weighted F1: 0.8835324208157846


In [ ]:
#  !pip install catboost


   ---------------------------------------- 0.0/102.5 MB ? eta -:--:--
   - -------------------------------------- 3.9/102.5 MB 29.4 MB/s eta 0:00:04
   ---- ----------------------------------- 10.7/102.5 MB 28.0 MB/s eta 0:00:04
   ------ --------------------------------- 17.0/102.5 MB 29.9 MB/s eta 0:00:03
   --------- ------------------------------ 23.3/102.5 MB 30.2 MB/s eta 0:00:03
   ---------- ----------------------------- 27.3/102.5 MB 27.9 MB/s eta 0:00:03
   ------------ --------------------------- 31.5/102.5 MB 25.9 MB/s eta 0:00:03
   ------------- -------------------------- 34.6/102.5 MB 24.1 MB/s eta 0:00:03
   -------------- ------------------------- 37.5/102.5 MB 23.1 MB/s eta 0:00:03
   ---------------- ----------------------- 41.4/102.5 MB 22.5 MB/s eta 0:00:03
   ----------------- ---------------------- 45.1/102.5 MB 21.9 MB/s eta 0:00:03
   ------------------- -------------------- 49.0/102.5 MB 21.5 MB/s eta 0:00:03
   -------------------- ------------------- 52.2/

In [ ]:
# %pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 15.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
#  pip install xgboost

   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   - -------------------------------------- 2.6/72.0 MB 16.9 MB/s eta 0:00:05
   ----- ---------------------------------- 10.0/72.0 MB 27.0 MB/s eta 0:00:03
   -------- ------------------------------- 16.0/72.0 MB 28.0 MB/s eta 0:00:03
   ------------ --------------------------- 22.0/72.0 MB 27.9 MB/s eta 0:00:02
   --------------- ------------------------ 27.8/72.0 MB 28.4 MB/s eta 0:00:02
   ------------------ --------------------- 34.1/72.0 MB 28.5 MB/s eta 0:00:02
   --------------------- ------------------ 38.8/72.0 MB 27.4 MB/s eta 0:00:02
   ----------------------- ---------------- 42.5/72.0 MB 26.2 MB/s eta 0:00:02
   -------------------------- ------------- 46.9/72.0 MB 25.3 MB/s eta 0:00:01
   --------------------------- ------------ 50.1/72.0 MB 24.3 MB/s eta 0:00:01
   ----------------------------- ---------- 53.0/72.0 MB 23.4 MB/s eta 0:00:01
   ------------------------------- -------- 56.1/72.0 MB 22.5 

In [ ]:
# Reuse the memory-friendly xgb_pipeline defined earlier (cell 16)
xgb_pipeline


In [ ]:
from sklearn.exceptions import NotFittedError
try:
    # fit only the preprocessor on training data to locate transformer error early
    preprocessor.fit(X_train)
    print("preprocessor.fit succeeded")
except Exception as e:
    print("preprocessor.fit failed with:")
    print(e)
    # Inspect X_train.sample for likely culprits
    print("\nSample rows that might cause problem:")
    print(X_train.sample(10))

In [ ]:
# show dtypes for columns used by the pipeline
print("Numeric columns:", numeric_cols)
print("Category columns:", category_cols)
print("Infra columns:", infra_cols)

print("\nDataFrame dtypes (columns of interest):")
print(df[numeric_cols + category_cols + infra_cols].dtypes)

# find which numeric column contains non-numeric values (samples)
for col in numeric_cols:
    if col in df.columns:
        # try to coerce a small sample; any non-convertible value will become NaN
        sample = df[col].dropna().astype(str).head(200)
        non_numeric = sample[~sample.apply(lambda v: v.replace('.', '', 1).lstrip('-').isdigit())]
        if len(non_numeric)>0:
            print(f"\nColumn {col} has non-numeric examples (showing up to 10):")
            print(non_numeric.unique()[:10])